# DermaMatch AI — Final Data Pipeline

## Ingredient-Aware Hybrid Skincare Recommendation Engine

This notebook prepares the **single final dataset** for the DermaMatch AI project:

> **Sephora Products and Skincare Reviews**

The pipeline is designed for the final architecture: **product catalog + reviews → cleaned skincare catalog → ingredient features + review-derived signals → recommendation-ready data → semantic documents → artifacts for Sentence Transformers / ChromaDB**.

### Final stack used by this data pipeline

- Python / Pandas / NumPy
- Scikit-learn utilities for text normalization and later evaluation
- Standard-library parsing for ingredients and deterministic review-theme extraction
- CSV artifacts for portability
- Sentence Transformers and ChromaDB are used in the downstream recommendation pipeline, not required for this preprocessing notebook

### Important design decision

We deliberately use **one source dataset only**. We do not add a separate ingredient dataset. Ingredient intelligence is extracted from the product ingredient field itself, while review data provides user-derived signals such as rating, recommendation rate, skin-type distribution, helpfulness, and recurring review themes.

## 1. Expected raw data layout

Place the extracted Kaggle dataset under:

```text
project-root/
├── data/
│   └── raw/
│       └── sephora/
│           ├── product_info.csv
│           ├── reviews_0-250.csv
│           ├── reviews_250-500.csv
│           ├── reviews_500-750.csv
│           ├── reviews_750-1250.csv
│           └── reviews_1250-end.csv
```

The code below also auto-detects the uploaded/extracted dataset in common local paths, so the notebook is less sensitive to where it is opened.

In [1]:
from pathlib import Path
import ast
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_RAW_DIRS = [
    Path("data/raw/sephora"),
    Path("../data/raw/sephora"),
    Path("/mnt/data/dermamatch_raw"),
    Path("/mnt/data/data/raw/sephora"),
]

RAW_DIR = next((p for p in CANDIDATE_RAW_DIRS if (p / "product_info.csv").exists()), None)
if RAW_DIR is None:
    raise FileNotFoundError(
        "product_info.csv was not found. Extract the Sephora dataset into "
        "data/raw/sephora/ and rerun this cell."
    )

PROJECT_ROOT = Path.cwd()
if RAW_DIR.is_absolute() and "/mnt/data" in str(RAW_DIR):
    OUTPUT_DIR = Path("dermamatch_processed")
else:
    OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DIR.resolve()}")
print(f"Processed output directory: {OUTPUT_DIR.resolve()}")

Raw data directory: D:\CODE\ORBO.ai\data\raw\sephora
Processed output directory: D:\CODE\ORBO.ai\data\processed


## 2. File discovery and raw schema inspection

The uploaded dataset contains one product catalog and multiple review CSVs. Review files are processed in **chunks** so the full review corpus does not need to reside in memory at once.

In [2]:
product_path = RAW_DIR / "product_info.csv"
review_paths = sorted(RAW_DIR.glob("reviews_*.csv"))

if not review_paths:
    raise FileNotFoundError("No reviews_*.csv files were found next to product_info.csv.")

print("Product file:", product_path.name)
print("Review files:")
for p in review_paths:
    print(" -", p.name)

products_raw = pd.read_csv(product_path)
print("\nProduct shape:", products_raw.shape)
print("Product columns:")
print(products_raw.columns.tolist())

review_preview = pd.read_csv(review_paths[0], nrows=5)
print("\nReview columns:")
print(review_preview.columns.tolist())

Product file: product_info.csv
Review files:
 - reviews_0-250.csv
 - reviews_1250-end.csv
 - reviews_250-500.csv
 - reviews_500-750.csv
 - reviews_750-1250.csv

Product shape: (8494, 27)
Product columns:
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']

Review columns:
['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']


## 3. Raw dataset profiling

Before cleaning, we inspect row counts, uniqueness, missingness, category distribution, ratings, and key join fields. The actual downloaded files are the source of truth for these values.

In [3]:
# Product-level profiling
product_profile = pd.DataFrame({
    "dtype": products_raw.dtypes.astype(str),
    "missing_count": products_raw.isna().sum(),
    "missing_pct": (products_raw.isna().mean() * 100).round(2),
    "n_unique": products_raw.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

display(product_profile)

print("Unique product_id:", products_raw["product_id"].nunique())
print("Duplicate product rows:", products_raw.duplicated().sum())
print("Duplicate product_id rows:", products_raw["product_id"].duplicated().sum())
print("\nPrimary categories:")
display(products_raw["primary_category"].value_counts(dropna=False).head(20))

,dtype,missing_count,missing_pct,n_unique
sale_price_usd,float64,8224,96.82,88
value_price_usd,float64,8043,94.69,174
variation_desc,object,7244,85.28,935
child_max_price,float64,5740,67.58,222
child_min_price,float64,5740,67.58,208
highlights,object,2207,25.98,4417
size,object,1631,19.20,2055
variation_value,object,1598,18.81,2729
variation_type,object,1444,17.00,7
tertiary_category,object,990,11.66,118


Unique product_id: 8494
Duplicate product rows: 0
Duplicate product_id rows: 0

Primary categories:


primary_category
Skincare           2420
Makeup             2369
Hair               1464
Fragrance          1432
Bath & Body         405
Mini Size           288
Men                  60
Tools & Brushes      52
Gifts                 4
Name: count, dtype: int64

In [4]:
# Count review rows without loading all reviews into memory.
review_counts = {}
review_rows_total = 0
for path in review_paths:
    row_count = 0
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        next(fh, None)  # header
        for _ in fh:
            row_count += 1
    review_counts[path.name] = row_count
    review_rows_total += row_count

review_file_profile = pd.DataFrame(
    {"file": list(review_counts), "rows": list(review_counts.values())}
)

display(review_file_profile)
print("Total review rows:", f"{review_rows_total:,}")

,file,rows
0,reviews_0-250.csv,610500
1,reviews_1250-end.csv,50694
2,reviews_250-500.csv,208906
3,reviews_500-750.csv,117175
4,reviews_750-1250.csv,120135


Total review rows: 1,107,410


## 4. Define the skincare universe

The source catalog contains makeup, hair, fragrance, bath & body, tools, gifts, and other categories. For this project, we restrict the recommendation catalog to products whose `primary_category` is **Skincare**.

This prevents unrelated products from entering the skincare recommendation engine.

In [5]:
products = products_raw.copy()
products["primary_category"] = products["primary_category"].fillna("").astype(str).str.strip()

SKINCARE_CATEGORY = "Skincare"
skincare = products[products["primary_category"].eq(SKINCARE_CATEGORY)].copy()

if skincare.empty:
    raise ValueError("No products with primary_category == 'Skincare' were found.")

print("All catalog products:", len(products))
print("Skincare products:", len(skincare))
print("Skincare share: {:.2%}".format(len(skincare) / len(products)))
print("\nSkincare secondary categories:")
display(skincare["secondary_category"].value_counts(dropna=False).head(30))

All catalog products: 8494
Skincare products: 2420
Skincare share: 28.49%

Skincare secondary categories:


secondary_category
Moisturizers              551
Treatments                466
Cleansers                 361
Value & Gift Sets         196
Eye Care                  186
Masks                     166
Mini Size                 112
Sunscreen                 108
High Tech Tools            80
Wellness                   79
Lip Balms & Treatments     61
Self Tanners               53
Shop by Concern             1
Name: count, dtype: int64

## 5. Product cleaning and normalization

We retain fields useful for recommendation, ranking, explainability, and UI display. Missing textual fields are converted to empty strings; numeric fields are coerced safely.

We also remove duplicate product IDs and keep the first canonical catalog row.

In [6]:
TEXT_COLUMNS = [
    "product_name", "brand_name", "size", "variation_type", "variation_value",
    "variation_desc", "ingredients", "highlights", "primary_category",
    "secondary_category", "tertiary_category"
]
NUMERIC_COLUMNS = [
    "loves_count", "rating", "reviews", "price_usd", "value_price_usd",
    "sale_price_usd", "child_count", "child_max_price", "child_min_price"
]
FLAG_COLUMNS = [
    "limited_edition", "new", "online_only", "out_of_stock", "sephora_exclusive"
]

for col in TEXT_COLUMNS:
    if col in skincare.columns:
        skincare[col] = skincare[col].fillna("").astype(str).str.strip()

for col in NUMERIC_COLUMNS:
    if col in skincare.columns:
        skincare[col] = pd.to_numeric(skincare[col], errors="coerce")

for col in FLAG_COLUMNS:
    if col in skincare.columns:
        skincare[col] = pd.to_numeric(skincare[col], errors="coerce").fillna(0).astype(int)

skincare = skincare.drop_duplicates(subset=["product_id"], keep="first").reset_index(drop=True)

# A stable, user-facing price: sale price if available, otherwise regular price.
skincare["effective_price_usd"] = skincare["sale_price_usd"].where(
    skincare["sale_price_usd"].notna(), skincare["price_usd"]
)

print("Clean unique skincare products:", len(skincare))
print("Missing effective price:", skincare["effective_price_usd"].isna().sum())
print("Missing ingredients:", (skincare["ingredients"] == "").sum())

Clean unique skincare products: 2420
Missing effective price: 0
Missing ingredients: 134


## 6. Deterministic ingredient parsing

The raw `ingredients` field is stored as string representations of lists and can contain multiple sub-products in gift sets. We parse it with safe standard-library parsing first, then normalize the text for downstream matching.

No external ingredient database is required for this final project.

In [7]:
def safe_list_from_string(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass
    return [text]


def clean_ingredient_text(value):
    parts = safe_list_from_string(value)
    text = " ".join(parts)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_ingredient_tokens(value):
    text = clean_ingredient_text(value).lower()
    if not text:
        return []
    # Remove common punctuation but preserve ingredient words and hyphens.
    text = re.sub(r"[^a-z0-9%+\- ]+", " ", text)
    tokens = [tok.strip("- ") for tok in text.split() if tok.strip("- ")]
    # Remove common non-ingredient headers that occur in gift-set ingredient blocks.
    stop_tokens = {"ingredients", "ingredient", "contains", "formula"}
    tokens = [t for t in tokens if t not in stop_tokens]
    return sorted(set(tokens))

skincare["ingredients_clean"] = skincare["ingredients"].map(clean_ingredient_text)
skincare["ingredient_tokens"] = skincare["ingredients"].map(extract_ingredient_tokens)
skincare["ingredient_count"] = skincare["ingredient_tokens"].str.len()

print(skincare[["product_name", "ingredients_clean", "ingredient_count"]].head(5).to_string(index=False))

                                             product_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

## 7. Product text document for semantic embeddings

This is the canonical text representation that the Sentence Transformer will embed later.

We intentionally combine product name, brand, category, highlights, description-like fields, price, and cleaned ingredients so semantic retrieval can connect natural-language user needs with information distributed across the catalog record.

In [8]:
def build_product_document(row):
    fields = [
        f"Product: {row.get('product_name', '')}",
        f"Brand: {row.get('brand_name', '')}",
        f"Category: {row.get('primary_category', '')}",
        f"Subcategory: {row.get('secondary_category', '')}",
        f"Type: {row.get('tertiary_category', '')}",
        f"Highlights: {row.get('highlights', '')}",
        f"Ingredients: {row.get('ingredients_clean', '')}",
    ]
    return " | ".join(x for x in fields if x.split(": ", 1)[-1].strip())

skincare["product_document"] = skincare.apply(build_product_document, axis=1)

print(skincare[["product_id", "product_name", "product_document"]].head(3).to_string(index=False))

product_id                            product_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

## 8. Review ingestion with memory-safe chunking

The review corpus is large, so we process each CSV in chunks. We keep only fields needed for recommendation signals:

- rating
- `is_recommended`
- helpfulness / feedback counts
- submission date
- review text / title
- skin type
- product ID

This avoids loading the full review corpus into RAM at once.

In [9]:
REVIEW_USECOLS = [
    "author_id", "rating", "is_recommended", "helpfulness",
    "total_feedback_count", "total_neg_feedback_count", "total_pos_feedback_count",
    "submission_time", "review_text", "review_title", "skin_type", "product_id"
]

# Validate required columns from a preview before starting the larger pass.
missing_review_cols = sorted(set(REVIEW_USECOLS) - set(review_preview.columns))
if missing_review_cols:
    raise ValueError(f"The review schema is missing required columns: {missing_review_cols}")

CHUNK_SIZE = 50_000
print(f"Using review chunk size: {CHUNK_SIZE:,}")

Using review chunk size: 50,000


## 9. Review theme extraction

The recommendation engine needs a deterministic review signal without depending on an external LLM. We therefore extract simple recurring themes from review text.

These themes are deliberately product-language features (for example `lightweight`, `greasy`, `hydrating`, `drying`, `fragrance`, `irritation`, `breakout`, and `texture`) rather than medical conclusions.

In [10]:
THEME_PATTERNS = {
    "lightweight": [r"\blightweight\b", r"\blight weight\b"],
    "greasy": [r"\bgreasy\b", r"\boily\b"],
    "hydrating": [r"\bhydrating\b", r"\bhydration\b", r"\bmoisturiz"],
    "drying": [r"\bdrying\b", r"\bdry\b"],
    "fragrance": [r"\bfragrance\b", r"\bscent\b", r"\bsmell\b"],
    "irritation": [r"\birritat", r"\bburning\b", r"\bsting", r"\bsensitive\b"],
    "breakout": [r"\bbreak ?out", r"\bacne\b", r"\bpimple", r"\bcomed"],
    "absorption": [r"\babsorbs?\b", r"\babsorbtion\b", r"\bsoak(s|ed|ing)?\b"],
    "sticky": [r"\bsticky\b", r"\btacky\b"],
    "texture": [r"\btexture\b", r"\bconsistency\b", r"\bfeel(s|ing)?\b"],
    "effective": [r"\bwork(s|ed)?\b", r"\beffective\b", r"\bresults?\b"],
}

COMPILED_THEMES = {
    theme: [re.compile(pattern, flags=re.IGNORECASE) for pattern in patterns]
    for theme, patterns in THEME_PATTERNS.items()
}


def normalize_review_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def theme_flags(text):
    text = str(text)
    return {
        theme: int(any(pattern.search(text) for pattern in patterns))
        for theme, patterns in COMPILED_THEMES.items()
    }

## 10. Review ingestion with memory-safe streaming

The review corpus is large, so the final pipeline uses Python's built-in `csv` reader and processes the files **row by row**. This keeps memory usage stable and avoids loading the full review corpus into RAM.

We also discard non-skincare reviews immediately using the `product_id` set from the cleaned skincare catalog.

In [11]:
from collections import defaultdict
import csv

SKINCARE_PRODUCT_IDS = set(skincare["product_id"].astype(str))

# Compact per-product accumulator. Each value is a dictionary of small numeric counters.
review_acc = defaultdict(lambda: {
    "review_count_observed": 0,
    "rating_sum": 0.0,
    "rating_n": 0,
    "recommendation_sum": 0.0,
    "recommendation_n": 0,
    "helpfulness_sum": 0.0,
    "helpfulness_n": 0,
    "review_length_sum": 0.0,
    "skin_counts": defaultdict(int),
    "theme_counts": defaultdict(int),
})

# Compile compact keyword patterns once.
THEME_REGEX = {
    theme: re.compile("|".join(patterns), flags=re.IGNORECASE)
    for theme, patterns in THEME_PATTERNS.items()
}

parsed_review_rows = 0
skincare_review_rows = 0
malformed_rows = 0

for path in review_paths:
    print(f"Processing {path.name} ...")
    with path.open("r", encoding="utf-8", errors="replace", newline="") as fh:
        reader = csv.reader(fh)
        header = next(reader, None)
        if header is None:
            continue
        col = {name: i for i, name in enumerate(header)}
        missing = [name for name in REVIEW_USECOLS if name not in col]
        if missing:
            raise ValueError(f"{path.name} is missing review columns: {missing}")

        for row in reader:
            parsed_review_rows += 1
            if len(row) < len(header):
                malformed_rows += 1
                continue

            product_id = row[col["product_id"]].strip()
            if product_id not in SKINCARE_PRODUCT_IDS:
                continue

            skincare_review_rows += 1
            item = review_acc[product_id]
            item["review_count_observed"] += 1

            rating_raw = row[col["rating"]].strip()
            try:
                rating = float(rating_raw) if rating_raw else np.nan
            except ValueError:
                rating = np.nan
            if np.isfinite(rating):
                item["rating_sum"] += rating
                item["rating_n"] += 1

            rec_raw = row[col["is_recommended"]].strip()
            try:
                rec = float(rec_raw) if rec_raw else np.nan
            except ValueError:
                rec = np.nan
            if np.isfinite(rec):
                item["recommendation_sum"] += rec
                item["recommendation_n"] += 1

            help_raw = row[col["helpfulness"]].strip()
            try:
                helpfulness = float(help_raw) if help_raw else np.nan
            except ValueError:
                helpfulness = np.nan
            if np.isfinite(helpfulness):
                item["helpfulness_sum"] += helpfulness
                item["helpfulness_n"] += 1

            title = row[col["review_title"]].strip()
            text = row[col["review_text"]].strip()
            combined = re.sub(r"\\s+", " ", f"{title}. {text}").strip(". ")
            item["review_length_sum"] += len(combined)

            skin_type = row[col["skin_type"]].strip().lower() or "unknown"
            item["skin_counts"][skin_type] += 1

            for theme, pattern in THEME_REGEX.items():
                if pattern.search(combined):
                    item["theme_counts"][theme] += 1

review_records = []
for product_id, item in review_acc.items():
    rec = {
        "product_id": product_id,
        "review_count_observed": item["review_count_observed"],
        "review_avg_rating": (
            item["rating_sum"] / item["rating_n"] if item["rating_n"] else np.nan
        ),
        "recommendation_rate": (
            item["recommendation_sum"] / item["recommendation_n"] if item["recommendation_n"] else np.nan
        ),
        "avg_helpfulness": (
            item["helpfulness_sum"] / item["helpfulness_n"] if item["helpfulness_n"] else np.nan
        ),
        "avg_review_length": (
            item["review_length_sum"] / item["review_count_observed"] if item["review_count_observed"] else 0.0
        ),
    }

    total = item["review_count_observed"]
    for skin_type, count in item["skin_counts"].items():
        rec[f"skin_share_{skin_type}"] = count / total if total else 0.0
    for theme in THEME_PATTERNS:
        rec[f"theme_{theme}_count"] = item["theme_counts"].get(theme, 0)
    review_records.append(rec)

review_signals = pd.DataFrame(review_records)

print(f"Parsed review rows: {parsed_review_rows:,}")
print(f"Skincare review rows retained: {skincare_review_rows:,}")
print(f"Malformed review rows skipped: {malformed_rows:,}")
print(f"Products with review signals: {len(review_signals):,}")

Processing reviews_0-250.csv ...
Processing reviews_1250-end.csv ...
Processing reviews_250-500.csv ...
Processing reviews_500-750.csv ...
Processing reviews_750-1250.csv ...
Parsed review rows: 1,094,411
Skincare review rows retained: 1,094,411
Malformed review rows skipped: 0
Products with review signals: 2,351


## 11. Inspect aggregated review signals

The streaming pass produces one compact row per skincare product with review-derived features.

In [12]:
expected_skin_types = ["oily", "dry", "combination", "normal", "sensitive"]

# Add explicit zero-valued columns where a skin type was not observed for a product.
for skin_type in expected_skin_types:
    col_name = f"skin_share_{skin_type}"
    if col_name not in review_signals.columns:
        review_signals[col_name] = 0.0

for col_name in [f"theme_{theme}_count" for theme in THEME_PATTERNS]:
    if col_name not in review_signals.columns:
        review_signals[col_name] = 0

review_signals = review_signals.sort_values("review_count_observed", ascending=False).reset_index(drop=True)

print("Review-signal columns:", len(review_signals.columns))
display(review_signals.head(10))

Review-signal columns: 23


,product_id,review_count_observed,review_avg_rating,recommendation_rate,avg_helpfulness,avg_review_length,skin_share_dry,theme_lightweight_count,theme_greasy_count,theme_hydrating_count,theme_drying_count,theme_fragrance_count,theme_irritation_count,theme_breakout_count,theme_absorption_count,theme_sticky_count,theme_texture_count,theme_effective_count,skin_share_unknown,skin_share_combination,skin_share_normal,skin_share_oily,skin_share_sensitive
0,P420652,16138,4.350477,0.829675,0.776157,289.290185,0.189553,52,220,3661,4993,2804,372,159,231,1085,4780,3147,0.044429,0.508365,0.137254,0.120399,0.0
1,P7880,8736,4.362637,0.832764,0.761184,296.781364,0.132669,53,838,641,2508,2488,2349,1477,5,27,3598,1531,0.244162,0.431090,0.107715,0.084364,0.0
2,P218700,7763,4.499807,0.859312,0.825964,363.619863,0.116063,379,2439,3166,2645,498,812,1871,1114,64,1937,1621,0.433209,0.315986,0.076388,0.058354,0.0
3,P248407,7547,4.518484,0.834325,0.815449,347.604479,0.241950,273,2040,3758,3787,854,1993,1272,871,154,2144,1864,0.272956,0.358818,0.076587,0.049689,0.0
4,P269122,7414,4.545050,0.940250,0.766417,308.117885,0.122336,1,344,365,708,223,1504,1740,46,27,2316,2694,0.194902,0.473294,0.107095,0.102374,0.0
5,P394639,7294,4.483685,0.897692,0.791261,292.779682,0.147381,1282,2356,5061,1996,1295,900,924,869,529,3525,1017,0.136002,0.485193,0.109953,0.121470,0.0
6,P450271,6169,4.495056,0.872846,0.807781,308.693792,0.196466,33,779,346,844,928,1235,568,15,39,2175,1189,0.039553,0.533150,0.114281,0.116550,0.0
7,P417238,6169,4.495056,0.872846,0.807781,308.693792,0.196466,33,779,346,844,928,1235,568,15,39,2175,1189,0.039553,0.533150,0.114281,0.116550,0.0
8,P427421,6063,3.960416,0.728352,0.764125,338.123701,0.191654,314,1333,3674,1432,326,906,1278,442,155,2409,1019,0.013195,0.557480,0.128154,0.109517,0.0
9,P411387,5864,4.209754,0.800705,0.786388,300.717940,0.171726,35,794,466,2043,1163,1100,1275,10,16,2658,913,0.038881,0.533424,0.120566,0.135402,0.0


## 12. Merge product and review signals

This is the central join in the final pipeline.

`product_id` is the canonical key. Products without reviews remain in the catalog so the recommender can still serve cold-start products; their review-based signals are filled with safe neutral defaults.

In [13]:
recommendation_catalog = skincare.merge(review_signals, on="product_id", how="left", suffixes=("", "_from_reviews"))

# Neutral/default values for products with no observed reviews.
for col in ["review_count_observed", "avg_review_length"] + [f"theme_{t}_count" for t in THEME_PATTERNS]:
    if col in recommendation_catalog.columns:
        recommendation_catalog[col] = recommendation_catalog[col].fillna(0)

for col in ["review_avg_rating", "recommendation_rate", "avg_helpfulness"]:
    if col in recommendation_catalog.columns:
        # Keep missing values when no reviews exist; downstream ranking can use catalog rating as fallback.
        recommendation_catalog[col] = pd.to_numeric(recommendation_catalog[col], errors="coerce")

for col in [c for c in recommendation_catalog.columns if c.startswith("skin_share_")]:
    recommendation_catalog[col] = recommendation_catalog[col].fillna(0)

# Review-derived rating fallback logic for downstream ranking.
recommendation_catalog["effective_rating"] = recommendation_catalog["review_avg_rating"].where(
    recommendation_catalog["review_avg_rating"].notna(), recommendation_catalog["rating"]
)

# Product has_reviews flag is useful for cold-start handling.
recommendation_catalog["has_reviews"] = (recommendation_catalog["review_count_observed"] > 0).astype(int)

print("Final recommendation catalog shape:", recommendation_catalog.shape)
print("Products with observed reviews:", recommendation_catalog["has_reviews"].sum())

Final recommendation catalog shape: (2420, 56)
Products with observed reviews: 2351


## 13. Product-level review theme features

Convert theme counts into per-review shares so heavily reviewed products do not automatically dominate merely because they have more reviews.

In [14]:
for theme in THEME_PATTERNS:
    count_col = f"theme_{theme}_count"
    share_col = f"theme_{theme}_share"
    recommendation_catalog[share_col] = (
        recommendation_catalog[count_col] / recommendation_catalog["review_count_observed"].replace(0, np.nan)
    ).fillna(0)

## 14. Create review-informed audience signals

The product catalog itself does not contain a product-level skin type field. Review records contain `skin_type`, so we derive product-level **review audience shares** such as:

- `skin_share_oily`
- `skin_share_dry`
- `skin_share_combination`
- `skin_share_normal`
- `skin_share_sensitive`

These are treated as **observed review audience signals**, not medical claims about who should use a product.

In [15]:
expected_skin_types = ["oily", "dry", "combination", "normal", "sensitive"]
for skin_type in expected_skin_types:
    col = f"skin_share_{skin_type}"
    if col not in recommendation_catalog.columns:
        recommendation_catalog[col] = 0.0

# A compact audience profile used later by the ranker.
recommendation_catalog["skin_type_profile"] = recommendation_catalog.apply(
    lambda row: ", ".join(
        f"{stype}: {row[f'skin_share_{stype}']:.2f}"
        for stype in expected_skin_types
        if row[f"skin_share_{stype}"] > 0
    ),
    axis=1,
)

## 15. Build the final recommendation document

The recommendation document contains the information needed for semantic candidate retrieval. Review-derived signals are included as compact text features rather than concatenating every raw review.

In [16]:
def pct(x):
    return f"{100 * x:.0f}%"


def build_recommendation_document(row):
    review_bits = []
    for theme in THEME_PATTERNS:
        share = row.get(f"theme_{theme}_share", 0.0)
        if share >= 0.10:
            review_bits.append(theme.replace("_", " "))

    audience_bits = []
    for skin_type in expected_skin_types:
        share = row.get(f"skin_share_{skin_type}", 0.0)
        if share >= 0.10:
            audience_bits.append(skin_type)

    parts = [
        f"Product: {row.get('product_name', '')}",
        f"Brand: {row.get('brand_name', '')}",
        f"Category: {row.get('primary_category', '')}",
        f"Subcategory: {row.get('secondary_category', '')}",
        f"Type: {row.get('tertiary_category', '')}",
        f"Highlights: {row.get('highlights', '')}",
        f"Ingredients: {row.get('ingredients_clean', '')}",
        f"Price USD: {row.get('effective_price_usd', '')}",
        f"Catalog rating: {row.get('rating', '')}",
        f"Observed review rating: {row.get('review_avg_rating', '')}",
        f"Review recommendation rate: {pct(row.get('recommendation_rate', 0.0)) if pd.notna(row.get('recommendation_rate', np.nan)) else ''}",
    ]
    if review_bits:
        parts.append("Recurring review themes: " + ", ".join(review_bits))
    if audience_bits:
        parts.append("Observed reviewer skin types: " + ", ".join(audience_bits))
    return " | ".join(x for x in parts if x.split(": ", 1)[-1].strip())

recommendation_catalog["recommendation_document"] = recommendation_catalog.apply(
    build_recommendation_document, axis=1
)

## 16. Data-quality checks before saving

These assertions are intentionally strict. If one fails, the data pipeline should stop rather than silently producing a corrupted recommendation catalog.

In [17]:
# Identity and join checks.
assert recommendation_catalog["product_id"].notna().all(), "product_id contains missing values."
assert recommendation_catalog["product_id"].is_unique, "product_id must be unique in the final product catalog."
assert recommendation_catalog["primary_category"].eq("Skincare").all(), "Non-skincare products entered the final catalog."

# Numeric sanity checks.
assert (recommendation_catalog["effective_price_usd"].dropna() >= 0).all(), "Negative price detected."
assert (recommendation_catalog["rating"].dropna().between(0, 5)).all(), "Catalog rating outside [0, 5]."
assert (recommendation_catalog["review_avg_rating"].dropna().between(0, 5)).all(), "Review rating outside [0, 5]."
assert (recommendation_catalog["recommendation_rate"].dropna().between(0, 1)).all(), "Recommendation rate outside [0, 1]."

# Text sanity checks.
assert recommendation_catalog["recommendation_document"].str.len().gt(0).all(), "Empty recommendation document detected."

# Share sanity checks.
share_cols = [c for c in recommendation_catalog.columns if c.startswith("skin_share_")]
for col in share_cols:
    assert recommendation_catalog[col].between(0, 1).all(), f"Invalid share in {col}."

print("All data-quality assertions passed.")

All data-quality assertions passed.


## 17. Save final processed artifacts

The notebook writes compact, reproducible artifacts used by the downstream recommendation service.

### Output files

- `skincare_products_clean.csv` — cleaned skincare product table
- `review_product_signals.csv` — aggregated review signals by product
- `recommendation_catalog.csv` — final merged recommendation catalog
- `ingredient_tokens.jsonl` — one product ID and normalized ingredient tokens per line
- `pipeline_manifest.json` — row counts, source files, and configuration

In [18]:
product_output = OUTPUT_DIR / "skincare_products_clean.csv"
review_output = OUTPUT_DIR / "review_product_signals.csv"
catalog_output = OUTPUT_DIR / "recommendation_catalog.csv"
ingredient_output = OUTPUT_DIR / "ingredient_tokens.jsonl"
manifest_output = OUTPUT_DIR / "pipeline_manifest.json"

# Select stable/important product fields for the clean product file.
product_columns = [
    "product_id", "product_name", "brand_id", "brand_name", "loves_count", "rating",
    "reviews", "size", "variation_type", "variation_value", "variation_desc",
    "ingredients", "ingredients_clean", "ingredient_count", "ingredient_tokens",
    "price_usd", "value_price_usd", "sale_price_usd", "effective_price_usd",
    "limited_edition", "new", "online_only", "out_of_stock", "sephora_exclusive",
    "highlights", "primary_category", "secondary_category", "tertiary_category",
    "child_count", "child_max_price", "child_min_price", "product_document"
]
product_columns = [c for c in product_columns if c in skincare.columns]
skincare_export = skincare[product_columns].copy()
skincare_export["ingredient_tokens"] = skincare_export["ingredient_tokens"].map(json.dumps)
skincare_export.to_csv(product_output, index=False)

review_signals.to_csv(review_output, index=False)

catalog_export = recommendation_catalog.copy()
catalog_export["ingredient_tokens"] = catalog_export["ingredient_tokens"].map(json.dumps)
catalog_export.to_csv(catalog_output, index=False)

with ingredient_output.open("w", encoding="utf-8") as f:
    for _, row in skincare[["product_id", "ingredient_tokens"]].iterrows():
        f.write(json.dumps({
            "product_id": row["product_id"],
            "ingredient_tokens": row["ingredient_tokens"],
        }, ensure_ascii=False) + "\n")

manifest = {
    "raw_directory": str(RAW_DIR.resolve()),
    "review_files": [p.name for p in review_paths],
    "product_rows_raw": int(len(products_raw)),
    "skincare_rows_final": int(len(skincare)),
    "review_rows_total": int(review_rows_total),
    "review_signal_rows": int(len(review_signals)),
    "recommendation_catalog_rows": int(len(recommendation_catalog)),
    "embedding_model_planned": "BAAI/bge-small-en-v1.5",
    "vector_database_planned": "ChromaDB",
    "backend_planned": "Flask",
    "frontend_planned": "Streamlit",
}
manifest_output.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:")
for p in [product_output, review_output, catalog_output, ingredient_output, manifest_output]:
    print(" -", p.resolve(), f"({p.stat().st_size / 1024 / 1024:.2f} MB)")

Saved:
 - D:\CODE\ORBO.ai\data\processed\skincare_products_clean.csv (9.20 MB)
 - D:\CODE\ORBO.ai\data\processed\review_product_signals.csv (0.44 MB)
 - D:\CODE\ORBO.ai\data\processed\recommendation_catalog.csv (13.35 MB)
 - D:\CODE\ORBO.ai\data\processed\ingredient_tokens.jsonl (1.99 MB)
 - D:\CODE\ORBO.ai\data\processed\pipeline_manifest.json (0.00 MB)


## 18. Final summary for the recommendation system

At the end of this notebook, the downstream recommender receives a **single clean product catalog** enriched with review-derived signals.

```text
Sephora raw product data
        +
Sephora raw reviews
        ↓
Cleaning + validation
        ↓
Skincare-only catalog
        ↓
Ingredient normalization
        ↓
Review aggregation
        ↓
Skin-type audience signals
        ↓
Review-theme signals
        ↓
Canonical recommendation document
        ↓
CSV artifacts
        ↓
Sentence Transformer embeddings
        ↓
ChromaDB
        ↓
Flask recommendation API
        ↓
Streamlit interface
```

### What this notebook intentionally does not do

- It does not fabricate medical suitability claims.
- It does not invent ingredient functions from an external source.
- It does not require loading all ~1M reviews into memory simultaneously.
- It does not create recommendation scores yet; those belong to the final ranking layer.
- It does not commit the raw Kaggle dataset to GitHub.

## 19. Next implementation stage

The next code component should consume `recommendation_catalog.csv` and implement the final recommendation engine:

1. Load the catalog.
2. Generate product embeddings with Sentence Transformers.
3. Index embeddings and metadata in ChromaDB.
4. Parse a quiz or natural-language request.
5. Retrieve Top-K semantic candidates.
6. Apply hard filters such as category, budget, and avoided terms.
7. Compute ingredient, review, preference, rating, and diversity features.
8. Re-rank candidates.
9. Produce deterministic explanations from the actual ranking signals.
10. Expose the result through Flask and Streamlit.